<a href="https://colab.research.google.com/github/d-noe/NLP_DH_PSL_Fall2025/blob/main/code/4_causal/Tutorial_4_LLM_Interaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative Language Models

This tutorial provides a quick overview of different methods to interact with LLMs through Python packages and API requests:
- [`transformers`](#transformers)
- [`ollama`](#ollama)
  - [`openai`](#openai) API library
  - [`requests`](#requests)

In the first part, we'll interact with LLMs via the `transformers` library. Moreover, we will use it as an excuse to examine [the influence of diverse generation parameters](#sampling) during inference (next-token prediction), and to disclose the [structure of a prompt](#prompt), i.e. what happens behind the hood when you prompt a chatbot.
Additionally, a quick example is provided to [automatize querying models](#automatize) for (large) sets of prompts (using batched generation to speed up the process).

Then, we'll use `ollama` (an open-source tool that allows to run open-weights LLMs on your local machine or server, please find their docs and additional information at [https://ollama.com/](https://ollama.com/)), to interact with LLMs. We'll deploy a local server that we will query through the API using [`openai`](#openai) Python package, or directly through [`requests`](#requests). While we will still be querying self-hosted LLMs, the same code could be used to interact with inference providers!

Note that both sections can be run independently.



![De La Serie (Des) Ordres. (detail) Vera Molnàr (1974).](https://dam.org/museum/wp-content/uploads/2020/10/MolnarDesOrdresConcentric-2000x2020.jpeg)
<p align="right">
  <i>(Des)Ordres</i>. Vera Molnàr (1974). Plotter drawing, ink on paper.<br>©VG Bild-Kunst, Bonn 2024.
</p>


<a name="transformers"></a>

# HuggingFace's `transformers` library


<details><summary>ℹ️ You could also directly use <tt>transformers.pipeline</tt></summary>

In the context of this notebook, we choose to interact with LLMs through the `generate` method of `AutoModelforCausalLM`s (mainly to dissect the different steps of inference, and to be able to inspect and potentially modify them), but `transformers` also provides `pipeline` classes that encapsulate whole processes, including (but not limited to) `'text-generation'`. You can find an example of how to use it below:

```python
from transformers import pipeline

# Main variables
model_name = "meta-llama/Llama-3-8b-instruct"    # from HuggingFace hub / or local        
batch_size = 8                                   # Number of prompts per batch (modify to fit your VRAM)
max_new_tokens = 10                              # Some generation arguments...

# Load pipeline -> it will handle everything!
generator = pipeline(      # (note: you can use `pipeline` for diverse tasks)
    "text-generation",     # state the task
    model=model_name,      # give the name of the LM to use
    device_map="auto",     # automatically moves model to GPU if available
)

# (Batched) Generation
outputs = generator(
    prompts,                       # list of prompts
    max_new_tokens=max_new_tokens, # you can give the pipeline some generation arguments
    batch_size=batch_size          # pipeline handles the batch internally
)

# Extract generated text
results = [o[0]["generated_text"] for o in outputs]

# Print
for i, txt in enumerate(results):
    print(f"\nPrompt {i+1}: {prompts[i]}")
    print(f"Generated: {txt}")

```

</details>


First, let's start with the installation and imports. Considering that we dispose of limited computational resources (make sure to use `GPU` runtime if you are running this notebook on Colab!), we will use the `bitsandbytes` library.

`bitsandbytes` is a lightweight CUDA library that enables efficient low-precision (8-bit and 4-bit) inference and training for LLMs: by quantizing model weights to fewer bits, it reduces GPU memory usage and often speeds up computation—making it possible to load and run models on hardware with limited resources without significant loss in performance.

Then we'll be able to load our model and tokenizer (using this config) as we've seen in earlier sessions, but here using `AutoModel`**`ForCausalLM`**.

In [ ]:
# allows to load quantized models (using less RAM, while preserving minimal performance loss)
!pip install -U bitsandbytes

In [ ]:
from transformers import (
    AutoTokenizer,              # Tokenizer
    AutoModelForCausalLM,       # ..CauselLM models -> transformer-based language generation
    BitsAndBytesConfig          # to load quantized versions of the models (use less RAM)
)
from torch import bfloat16
import torch

In [ ]:
# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # 4-bit quantization
    bnb_4bit_quant_type='nf4',       # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

In [ ]:
# Quantization config: load a large model using much less GPU memory.
# bitsandbytes handles the low-precision math under the hood.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # Store model weights in 4-bit precision instead of 16/32-bit.
                                      # This reduces memory footprint and allows larger models to fit on a single GPU.

    bnb_4bit_quant_type='nf4',       # Type of quantization: Use NF4 ("Normalized Float 4") quantization, a 4-bit format
                                      # designed to preserve model accuracy better than standard int4.

    bnb_4bit_use_double_quant=True,  # Apply a second layer of quantization to the quantization constants themselves.
                                      # This further reduces memory use with minimal extra overhead.

    bnb_4bit_compute_dtype=bfloat16  # Computation precision: Use bfloat16 for actual computation (matmul, attention, etc.).
                                      # This keeps numerical stability while keeping memory low (weights are in 4-bit format, but computation in 16).
                                      # bfloat16 -> 1 bit sign + 8 bits exponent + 7 bits mantissa
)

<a name="sampling"></a>

## Sampling & Generation Parameters

To start with, we'll use a 'base' (next-token predictor) model, here: ["gpt2"](https://huggingface.co/openai-community/gpt2) (but you can choose any other model, as long as it fits in memory).
This is an example of a model that is released after the "pre-training" stage but that did not go through the alignment (SFT + RL) steps.

In [ ]:
# Note: we're using 'base', purely next-token prediction version here
model_name = "gpt2" # Name of the model from HF hub

In [ ]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name                                  # name of the model to load associated tokenizer
)
# load LM
model = AutoModelForCausalLM.from_pretrained(   # type of LM: AutoModelForCausalLM
    model_name,                                 # name of the model
    quantization_config=bnb_config              # apply the quantization config
)

In [ ]:
# @title Helpers
# @markdown Some methods to visualize LLM's output probability distribution over the vocabulary.

import matplotlib as mpl
import matplotlib.pyplot as plt

def get_probabilities_from_logits(logits: torch.Tensor) -> torch.Tensor:
    """
    Takes the final-step logits from a model and converts them to probabilities.
    logits: shape (..., vocab_size). Typically: model_output.logits[:, -1, :]
    """
    return torch.softmax(logits, dim=-1)


def get_top_n_tokens(probs: torch.Tensor, n: int):
    """
    Returns top-n token IDs and their probabilities.
    probs: probability vector of shape (vocab_size,)
    """
    top_probs, top_ids = torch.topk(probs, n)
    return top_ids, top_probs


def decode_tokens(token_ids: torch.Tensor, tokenizer=None):
    """
    Decode token IDs to strings; if tokenizer is None, return numeric IDs as strings.
    """
    if tokenizer:
        return [tokenizer.decode([tid]) for tid in token_ids]
    return [str(tid.item()) for tid in token_ids]


def plot_probability_barchart(token_labels, token_probs, colormap="copper"):
    """
    Plot a bar chart for tokens and their probabilities.
    """
    cmap = mpl.colormaps[colormap]
    colors = cmap(token_probs / token_probs.max())

    plt.figure(figsize=(10, 4))
    #plt.bar(token_labels, token_probs)
    plt.bar(token_labels, token_probs, color=colors)
    plt.ylabel("Probability")
    plt.xlabel("Token")
    plt.title("Top Token Probabilities")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Below, we'll see how to use the model for next-token prediction and we'll play with some generation parameters to observe their influence on next-token's probability distribution.

As a reminder, a shortlist of generation arguments is provided in the table below:

| Parameter                                  | Description                                  | Typical Range                      |
| ------------------------------------------ | -------------------------------------------- | ---------------------------------- |
| `temperature`                              | Controls randomness (higher = more creative) | ]0.0 – $\infty$[                          |
| `top_p`                                    | Nucleus sampling; probability mass to keep   | [0. – 1.0]                          |
| `max_new_tokens`                              | Maximum tokens to generate                   | [0, $\infty$[ ;  e.g., 1, 10, 256, 512, ...                     |
| `stop_strings`                                     | Stop sequence(s)                             | list of strings; e.g. `["\nUser:", "\nAssistant:"]` |
| `repetition_penalty`                           | Penalize repeating tokens                    | 1.0 – 2.0                          |

<br><br>
But first, we define a (incomplete) piece of text that we'd like the model to continue:

In [ ]:
# @title Define your text for next token-prediction:

text = "the cat sat on the"

And... we can now generate text based to continue our sentence:

In [ ]:
# Tokenize the text and put it on the same device as the model
tokenized = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate next token(s) prediction
outputs = model.generate(
    **tokenized,
    max_new_tokens=20,     # maximal number of tokens to generate
)

model_completion = outputs[0][tokenized["input_ids"].shape[-1]:]  # at the position of the generated text (after length of input)
decoded_completion = tokenizer.decode(model_completion)           # decode the token IDs generated by the model into associated strings

print(f"[Our input:] {text}\n")
print(f"[The model completion:] {decoded_completion}\n")

print(f"------------\nAll together:\n\n[{text}][{decoded_completion}]")


Remember that text generation with LLMs is an iterative (auto-regressive) process. Behind the hood, the model generates tokens one-at-a-time: meaning that it first generate one token, then add it to the original input sequence to generate the next token, etc. untill reaching the maximal number of tokens (or the end generation).

So let's just focus on the very next token, and see how different generation parameters (primarily focusing on `temperature`, `top_p`, and `top_k`) impact the probability distribution in the model's output probability distribution.

In [ ]:
# @title Next token prediction:`⍰`
# @markdown Before going further, let's see the probability distribution for the next token, using 'default' generation arguments (here: `T=1` (no impact over logits), `top_p=1.` (no filtering before sampling)).

# plot top 10 tokens
n_plot = 10

tokenized = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Get logits through model.generate
outputs = model.generate(
    **tokenized,
    max_new_tokens=1,
    do_sample=True,               # to allow temperature / top-p / top-k
    temperature=1.,
    top_p=1.,
    top_k=n_plot+1,
    return_dict_in_generate=True, # to retrieve logits from output
    output_scores=True,
)

logits = outputs.scores[0]

probs = get_probabilities_from_logits(logits)[0]

top_ids, top_probs = get_top_n_tokens(probs, n=n_plot)
labels = decode_tokens(top_ids, tokenizer=tokenizer)

plot_probability_barchart(labels, top_probs.detach().cpu())

In [ ]:
# @title Next token prediction:`⍰` | **T = 0.2**
# @markdown Then with a lower temperature...
# @markdown
# @markdown Reminder: the temperature $T$ is used to weight the logits, following: $P_i=\frac{e^{\frac{z_i}{T}}}{\sum e^{\frac{z_j}{T}}}$.

tokenized = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Get logits through model.generate
outputs = model.generate(
    **tokenized,
    max_new_tokens=1,
    do_sample=True,               # to allow temperature / top-p / top-k
    temperature=.2,
    top_p=.9,
    top_k=n_plot+1,
    return_dict_in_generate=True, # to retrieve logits from output
    output_scores=True,
)

logits = outputs.scores[0]

probs = get_probabilities_from_logits(logits)[0]

top_ids, top_probs = get_top_n_tokens(probs, n=n_plot)
labels = decode_tokens(top_ids, tokenizer=tokenizer)

plot_probability_barchart(labels, top_probs.detach().cpu())

In [ ]:
# @title Next token prediction:`⍰` | **T = 5**
# @title Next token prediction:`⍰` | **T = 0.2**
# @markdown ... and with a higher temperature!
# @markdown
# @markdown Reminder: the temperature $T$ is used to weight the logits, following: $P_i=\frac{e^{\frac{z_i}{T}}}{\sum e^{\frac{z_j}{T}}}$.

tokenized = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Get logits through model.generate
outputs = model.generate(
    **tokenized,
    max_new_tokens=1,
    do_sample=True,               # to allow temperature / top-p / top-k
    temperature=5.,
    top_p=.9,
    top_k=n_plot+1,
    return_dict_in_generate=True, # to retrieve logits from output
    output_scores=True,
)

logits = outputs.scores[0]

probs = get_probabilities_from_logits(logits)[0]

top_ids, top_probs = get_top_n_tokens(probs, n=n_plot)
labels = decode_tokens(top_ids, tokenizer=tokenizer)

plot_probability_barchart(labels, top_probs.detach().cpu())

In [ ]:
# @title Next token prediction:`⍰` | **top_p = 0.6** (and T = 0.9)

# @markdown With a lower 'top p'
# @markdown
# @markdown Reminder: top p allows to keep only the smallest set of tokens whose cumulative probability ≥ top_p.


tokenized = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Get logits through model.generate
outputs = model.generate(
    **tokenized,
    max_new_tokens=1,
    do_sample=True,               # to allow temperature / top-p / top-k
    temperature=.9,
    top_p=.6,
    top_k=n_plot+1,
    return_dict_in_generate=True, # to retrieve logits from output
    output_scores=True,
)

logits = outputs.scores[0]

probs = get_probabilities_from_logits(logits)[0]

top_ids, top_probs = get_top_n_tokens(probs, n=n_plot)
labels = decode_tokens(top_ids, tokenizer=tokenizer)

plot_probability_barchart(labels, top_probs.detach().cpu())

In [ ]:
# @title Next token prediction:`⍰` | **top_k = 4** (and T = 0.9)

# @markdown With a lower 'top k'
# @markdown
# @markdown Reminder: top k keeps only the 'k' most probable tokens.


tokenized = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Get logits through model.generate
outputs = model.generate(
    **tokenized,
    max_new_tokens=1,
    do_sample=True,               # to allow temperature / top-p / top-k
    temperature=.9,
    top_p=.9,
    top_k=4,
    return_dict_in_generate=True, # to retrieve logits from output
    output_scores=True,
)

logits = outputs.scores[0]

probs = get_probabilities_from_logits(logits)[0]

top_ids, top_probs = get_top_n_tokens(probs, n=n_plot)
labels = decode_tokens(top_ids, tokenizer=tokenizer)

plot_probability_barchart(labels, top_probs.detach().cpu())

<a name="prompt"></a>

## Elements of a Prompt

Let's explore what's going on under the hood when interacting with chatbots. We'll dissect the prompt format used by such models, here using Ai2's [OLMo Instruction-tuned model](https://huggingface.co/allenai/OLMo-2-0425-1B-Instruct):

In [ ]:
# Note: we're using 'Instruct' version here, not 'base' purely next-token prediction version
model_name = "allenai/OLMo-2-0425-1B-Instruct" # Name of the model from HF hub

In [ ]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name                                  # name of the model to load associated tokenizer
)
# load LM
model = AutoModelForCausalLM.from_pretrained(   # type of LM: AutoModelForCausalLM
    model_name,                                 # name of the model
    quantization_config=bnb_config              # apply the quantization config
)

Now we write our prompt in a format that can easily be encoded by tokenizer and best understood by the model:

```python
prompt = [       # list of dicts -> chat history
  {              # each dict contains:
    "role": str,    # 'role' -> one of: "system" | "assistant" | "user"
    "content": str, # 'content': actual message
  }
]
```

Let's do it in practice, and try to interact with the model! First we define our prompt according to the chat-format:

In [ ]:
messages = [
    {
			"role": "user",
			"content": "Who are you?",
		},
]

Then, as usual, we tokenize our prompt. Yet, this time, in addition to standard tokenization, we ask the tokenizer to apply the chat template associated with the model (interaction we'll be more fluid as this is the format the model was trained with), and to add a 'generation_prompt' that will *prompt* the model to respond:

In [ ]:
str_tokenized_messages = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=False,
	return_dict=False,
	return_tensors="pt",
)

print(str_tokenized_messages)

As you can observe, the tokenizer wrapped our prompt into a format that will be more efficient to interact with the model using special tokens (standard start of text tokens, but also special tokens to indicate the 'role' of the conversational partners, and *prompt* the model to respond by including its role at the end, which means: this is your turn).

Then we convert it into model's vocabulary, i.e. token IDs (note, we can also do it in one go):

In [ ]:
inputs = tokenizer(
	str_tokenized_messages,
	return_tensors="pt",
).to(model.device)

Now we can input it into the model and get its response!:

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=126      # maximum number of tokens generated
)
model_response = outputs[0][inputs["input_ids"].shape[-1]:]
decoded_response = tokenizer.decode(model_response)

print(f"[USER MESSAGE  :] {messages[0]['content']}")
print(f"[MODEL RESPONSE:] {decoded_response}")

That's it! ✨

We can now try more interesting prompts, and even have multi-turn conversations with the model:

In [ ]:
messages = [
    {
			"role": "user",
			"content": "In the sentence 'The doctor married the nurse because she was pregnant.', who is pregnant?",
		},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    top_p=0                 # deterministic output
)

model_response = outputs[0][inputs["input_ids"].shape[-1]:]
decoded_response = tokenizer.decode(model_response)

print(f"[USER MESSAGE  :] {messages[0]['content']}")
print(f"[MODEL RESPONSE:] {decoded_response}")

For a multiturn conversations (including a system prompt), it would look something like:
```python
prompt = [       
  {              
    "role": "system",    
    "content": "SYSTEM_PROMPT",
  {              
    "role": "user",    
    "content": "USER_PROMPT_1",
  }, {             
    "role": "assistant",    
    "content": "LLM_RESPONSE_1",
  }, {
    "role": "user",    
    "content": "USER_PROMPT_2",
  }, {
    "role": "assistant",    
    "content": "LLM_RESPONSE_2",
  }, {
    "role": "user",    
    "content": "USER_PROMPT_3",
  }, {
    "role": "assistant",    
    "content": "LLM_RESPONSE_3",
  },
  # ...
]
```

Note: if you'd fill the utterances fields iteratively field during conversation, you would have a chatbot!

For now, let's just simulate a multi-turn conversation:

In [ ]:
messages = [ # Simulate multi-turn conversation
    {
			"role": "user",
			"content": "In the sentence 'The doctor married the nurse because she was pregnant.', who is pregnant?",
		},
    {
      "role": "assistant",  # ~ answer that I got earlier (but it doesn't need to be...)
			"content": """The sentence "The doctor married the nurse because she was pregnant." implies that the pregnant person is the nurse.""",
		},
    {
      "role": "user",
			"content": "In the previous sentence, the nurse is a man, then who is pregnant?",
		},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    top_p=0
)

model_response = outputs[0][inputs["input_ids"].shape[-1]:]
decoded_response = tokenizer.decode(model_response)

print("--------- CHAT HISTORY\n")
for m in messages:
	if m["role"] == "user":
		print(f"[USER MESSAGE  :] {m['content']}")
	elif m["role"] == "assistant":
		print(f"[MODEL RESPONSE:] {m['content']}")
print("\n--------- NEW RESPONSE\n")

print(f"[MODEL RESPONSE:] {decoded_response}")

<a name="automatize"></a>

## [Bonus] Automatize the process and prompt in batches

Note that we can use this syntax to automatize the process and run batches of prompts to get models' answers for several prompts.

Below is an example for using LLMs to predict/annotate data (zero-shot), here applied to the 'canonicity prediction' task (warning, it might be heavy to run it on Colab...):

In [ ]:
from tqdm import tqdm

In [ ]:
# helpers to load the data
try:
  from helpers import load_csv_from_github, load_dataset_from_github
except:
  !wget https://raw.githubusercontent.com/d-noe/NLP_DH_PSL_Fall2025/refs/heads/main/code/scripts/helpers.py
  from helpers import load_csv_from_github, load_dataset_from_github

In [ ]:
#@title Load the dataset of interest

dataset = load_dataset_from_github("data/canon_challenge/dataset")
dataset

In [ ]:
#@title Define chat and prompt template

# The prompt template we will use
# (just a quick example, we could write it in many different ways
#  e.g., - in the language of the data (here French if working on the 'canonicity prediction')
#        - providing additional instructions or reasoning cues
#.       - adding examples, for one-/few- shot classifiction, from the 'train' split
#.       - ....
# ):
prompt_template = """I will show you an excerpt of a book, and you have to tell me wether it is extracted from a book that belongs to the 'literary canon' or not.
Please answer only with one word:
Answer "Yes" if the text is extracted from a book that belongs to the canon.
Answer "No" if the text is extracted from a book that DOES NOT belong to the canon.

Here is the excerpt:
  {example}
"""

# We can also add a 'system_prompt':
system_prompt = "You are an expert in literary studies. Books that belong to the literary canon have no secret for you. You directly recognize if a sentence was extracted from a book that belongs to the canon."

prompts = [ # make one prompt per instance in the dataset's 'test' split
    [
        { # our system prompt
            "role":"system",
            "content": system_prompt
        },
        { # our template prompt, including the text chunk
            "role":"user" ,
            "content":prompt_template.format(example=ex["text"])
        }
    ]
    for ex in dataset["test"] # -> for each instance of the test split
]

In [ ]:
#@title Generate responses in batches

batch_size = 16               # user-defined batch size (make sure it fits into memory)
max_new_tokens = 10           # you can define additional generation arguments

# --- Helper function for batching ---
def batch_chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

# --- Batched generation ---
generated_texts = []

for batch_prompts in tqdm(batch_chunks(prompts, batch_size)):
    # Tokenize batch
    # tokenize the prompts + apply chat template for interacting with the model
    inputs = tokenizer.apply_chat_template(
        batch_prompts,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        padding_side='left'
    ).to(model.device)

    # Generate outputs
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decode and store results
    batch_texts = tokenizer.batch_decode(outputs[:,inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    generated_texts.extend(batch_texts)

In [ ]:
#@title Parse the answers into your desired format
#@markdown You can devise more advanced parsing mechanism or adapt it to your prompt instructions

predictions = [
    0 if ('yes' in resp.lower()) else 1
    for resp in generated_texts
]

<a name="ollama"></a>

# Run models locally with Ollama

The following cells are specifically written to run `ollama` in `Colab` environment.

If not (e.g. running on your personal machine), please consider the following:

> If running this notebook on your machine, you can still use the code below, or follow the instructions in ollama's official documentation: see [https://docs.ollama.com/quickstart](https://docs.ollama.com/quickstart) to download ollama and install a model of your choice. You will then be able to use it in your terminal through the CLI:
> ```bash
> ollama run <MODEL_NAME>
> ```
> And you can chat with the model, and deploy it on a local server! (using CLI: `ollama serve`).
>
> Then, you directly [jump to model interaction examples](#openai).
>
> ⚠️ Please note that—if running locally— this will install `ollama` on your machine and download models locally! ⚠️


Let's set up `ollama`. We start by downloading the software:

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

We can now run an `ollama` server in the background (either in terminal, or here using `subprocess` python library):

In [ ]:
import subprocess

proc = subprocess.Popen(["ollama", "serve"]) # Run it in the background
import time

time.sleep(3)  # Wait for a few seconds for Ollama to load!

Now we choose a model from [`ollama` library](https://ollama.com/library). Then we'll download it so that we can interact with it later!

In [ ]:
ollama_model_name = "llama3.2:1b"

In [ ]:
!ollama pull $ollama_model_name # Downloaed model

We can see the list of download models from `ollama` with:

In [ ]:
!ollama list # print the list of downloaded models

... and we can interact with these models! E.g. with cURL:

In [ ]:
!curl http://localhost:11434/api/chat -d '{ \
  "model": "llama3.2:1b", \
  "messages": [ \
      { "role": "user", "content": "Continue the Fibonacci sequence: 1, 1, 2, 3, 5, 8," } \
  ] \
}'

Now, let's see how to interact with LLMs through APIs inference providers: using [`openai` Python package](#openai), or directly via [`requests`](#requests).

> ℹ️ Note:
>
> In the following we'll be using our locally-hosted model running with `ollama`, but it the same syntax would apply if querying from a providers' endpoint (just changing the `base_url`).

<a name="openai"></a>

## Python `openai` API library

In [ ]:
from openai import OpenAI

In [ ]:
client = OpenAI(                            # instantiate the client using OpenAI package
    base_url='http://localhost:11434/v1/',  # local port (could be replaced with providers API endpoint!)
    api_key='ollama',                       # required but ignored (you should put your SECRET token / API key here if querying models through providers APIs)
)

We can now interact with the LLM through the client and chat with it:

In [ ]:
ollama_model_name = "llama3.2:1b" # name of the model to interact with

In [ ]:
# Side note: see the cultural *prompting* in the following examples? 🏈 or ⚽️ ..?

chat_completion = client.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': "What is the color of a football ball?",
        }
    ],
    model=ollama_model_name,
)

print(f'[Output :] {chat_completion.choices[0].message.content}')

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': "What is the colour of a football ball?",
        }
    ],
    model=ollama_model_name,
)

print(f'[Output :] {chat_completion.choices[0].message.content}')

<a name="requests"></a>

## Direct `requests`

Alternatively, we can use the `requests` Python package.

`requests` is a simple and user-friendly Python package for making HTTP requests. It abstracts away low-level networking logic and lets you interact with web APIs using clean, readable code.

In [ ]:
import requests

resp = requests.post(
    url="http://localhost:11434/v1/chat/completions",   # reach directly the chat completion endpoint
    json={
      "model": "llama3.2:1b",
      "messages": [
          {"role":"user", "content":"Write a haiku about Zinedine Zidane"}
      ],
      "temperature":0,
})
print(resp.json()["choices"][0]["message"]["content"])